In [43]:
import sys
import importlib
sys.path.append('../')  # Adjust the path as needed

import utilities.functions as functions
import utilities.plot as plot

# Reload the module to reflect the changes
importlib.reload(functions)
importlib.reload(plot)

<module 'utilities.plot' from '/Users/xuechenkan/potts_model_test/ms1/../utilities/plot.py'>

In [44]:
IN_seq_path = 'IN/data/in.reduce4.seq'
PR_seq_path = 'PR/data/pr.exper.reduce4.seq'    
RT_seq_path = 'RT/data/rt.reduce4.seq'

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_consensus = 'IN/data/in.consensus.reduce4.seq'
with open(IN_consensus, 'r') as f:
    IN_consensus_seq = f.read().strip()
# print("IN consensus sequence:", IN_consensus_seq)
PR_consensus = 'PR/data/pr.consensus.reduce4.seq'
with open(PR_consensus, 'r') as f:
    PR_consensus_seq = f.read().strip()
RT_consensus = 'RT/data/rt.consensus.reduce4.seq'
with open(RT_consensus, 'r') as f:
    RT_consensus_seq = f.read().strip()

# print(IN_consensus_seq)
    
IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux',1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux',0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux',0)

IN_J = functions.load_J_dict('IN/data/J.npy',1,263)
PR_J = functions.load_J_dict('PR/data/J_PR.npy',1,99)
RT_J = functions.load_J_dict('RT/data/J_RT.npy',39,226)

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')
RT_all_seq_unreduced = functions.read_seq('RT/data/rt.fullseq')

In [46]:
import csv

# Define the integrase mutation pairs
IN_pairs = ['G140S-Q148H', 'Y143C-S230R', 'G140A-Q148K', 'G140S-Q148R', 'G140S-Q148K', 'G140A-Q148R', 'E138K-Q148K', 'G140A-Q148H', 'E138K-Q148R', 'Y143C-S230K']
# Define the epistasis subsets
epistasis_subsets = ['Gain_of_function', 'rescue', 'rescue+gof', 'non_compensatory']

# Prepare the data for the CSV
csv_data = []

for IN_pair in IN_pairs:
    pair1, pair2 = functions.split_pairs(IN_pair)
    p1_reduced = functions.unreduced_to_reduced(IN_redux, pair1)
    p2_reduced = functions.unreduced_to_reduced(IN_redux, pair2)
    wt1, pos1, mt1 = functions.split_pair(p1_reduced)
    wt2, pos2, mt2 = functions.split_pair(p2_reduced)

    ########
    gof_counts = 0
    gof_without_DMC_count = 0
    gof_with_DMC_count = 0
    gof_with_one_mut_count = 0

    ########
    rescue_count = 0
    rescue_without_DMC_count = 0
    rescue_with_DMC_count = 0
    rescue_with_one_mut_count = 0

    ########
    compensatory_counts = 0
    compensatory_without_DMC_count = 0
    compensatory_with_DMC_count = 0
    compensatory_with_one_mut_count = 0

    ########
    noncompensatory_counts = 0
    noncompensatory_without_DMC_count = 0
    noncompensatory_with_DMC_count = 0
    noncompensatory_with_one_mut_count = 0

    gof_seqs = []
    rescue_seqs = []
    noncomp_seqs = []

    for IN_seq in IN_all_seq:
        result_merged = functions.calculate_dde_v2(p1_reduced, p2_reduced, IN_seq, IN_J, 1, 263)

        if result_merged is None:
            continue
        pair1_de, pair2_de, pair12_de, pair12_dde = result_merged

        if pair1_de < pair12_de and pair2_de < pair12_de and pair12_de > 0:
            gof_counts += 1
            gof_seqs.append(IN_seq)
            if IN_seq[pos1 - 1] == wt1 and IN_seq[pos2 - 1] == wt2:
                gof_without_DMC_count += 1
            elif IN_seq[pos1 - 1] == mt1 and IN_seq[pos2 - 1] == mt2:
                gof_with_DMC_count += 1
            else:
                gof_with_one_mut_count += 1

        elif pair1_de < pair12_de and pair2_de < pair12_de:
            rescue_count += 1
            rescue_seqs.append(IN_seq)
            if IN_seq[pos1 - 1] == wt1 and IN_seq[pos2 - 1] == wt2:
                rescue_without_DMC_count += 1
            elif IN_seq[pos1 - 1] == mt1 and IN_seq[pos2 - 1] == mt2:
                rescue_with_DMC_count += 1
            else:
                rescue_with_one_mut_count += 1

        elif pair1_de < pair12_de or pair2_de < pair12_de:
            compensatory_counts += 1
            if IN_seq[pos1 - 1] == wt1 and IN_seq[pos2 - 1] == wt2:
                compensatory_without_DMC_count += 1
            elif IN_seq[pos1 - 1] == mt1 and IN_seq[pos2 - 1] == mt2:
                compensatory_with_DMC_count += 1
            else:
                compensatory_with_one_mut_count += 1
        else:
            noncompensatory_counts += 1
            noncomp_seqs.append(IN_seq)
            if IN_seq[pos1 - 1] == wt1 and IN_seq[pos2 - 1] == wt2:
                noncompensatory_without_DMC_count += 1
            elif IN_seq[pos1 - 1] == mt1 and IN_seq[pos2 - 1] == mt2:
                noncompensatory_with_DMC_count += 1
            else:
                noncompensatory_with_one_mut_count += 1

    # Calculate average probabilities
    p_SH_gofs = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, IN_J, 1, 263) for seq in gof_seqs]
    average_p_gof = sum(p_SH_gofs) / len(p_SH_gofs) if p_SH_gofs else 0

    p_SH_rescues = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, IN_J, 1, 263) for seq in rescue_seqs]
    average_p_rescue = sum(p_SH_rescues) / len(p_SH_rescues) if p_SH_rescues else 0

    average_p_gof_rescue = (sum(p_SH_gofs) + sum(p_SH_rescues)) / (len(p_SH_gofs) + len(p_SH_rescues)) if (len(p_SH_gofs) + len(p_SH_rescues)) > 0 else 0

    p_SH_noncomps = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, IN_J, 1, 263) for seq in noncomp_seqs]
    average_p_noncomp = sum(p_SH_noncomps) / len(p_SH_noncomps) if p_SH_noncomps else 0

    # Calculate the actual_p values
    gof_actual_p = gof_with_DMC_count / gof_counts if gof_counts > 0 else 0
    rescue_actual_p = rescue_with_DMC_count / rescue_count if rescue_count > 0 else 0
    gof_rescue_actual_p = (gof_with_DMC_count + rescue_with_DMC_count) / (gof_counts + rescue_count) if (gof_counts + rescue_count) > 0 else 0
    noncomp_actual_p = noncompensatory_with_DMC_count / noncompensatory_counts if noncompensatory_counts > 0 else 0

    # Append data for each epistasis subset
    csv_data.append([IN_pair, 'Gain_of_function', gof_counts, average_p_gof, gof_actual_p])
    csv_data.append([IN_pair, 'rescue', rescue_count, average_p_rescue, rescue_actual_p])
    csv_data.append([IN_pair, 'rescue+gof', gof_counts + rescue_count, average_p_gof_rescue, gof_rescue_actual_p])
    csv_data.append([IN_pair, 'non_compensatory', noncompensatory_counts, average_p_noncomp, noncomp_actual_p])

# Write the data to a CSV file
with open('integrase_subset_probabilities.csv', 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow(['mutation_pair', 'epistasis_subset', 'num_seqs', 'average_p', 'actual_p'])
    csv_writer.writerows(csv_data)

print("CSV file 'integrase_subset_probabilities.csv' created successfully.")


CSV file 'integrase_subset_probabilities.csv' created successfully.


In [47]:
import csv

PR_pairs = ['D30N-N88D', 'V32I-I47V', 'G48V-I54A', 'D30N-K45Q', 'I54A-V82A']
PR_pairs.extend(['I54V-V82A', 'M46I-L76V', 'I54V-V82T', 'I54A-V82T', 'G48V-V82A'])
# Define the epistasis subsets
epistasis_subsets = ['Gain_of_function', 'rescue', 'rescue+gof', 'non_compensatory']

# Prepare the data for the CSV
csv_data = []

for PR_pair in PR_pairs:
    pair1, pair2 = functions.split_pairs(PR_pair)
    p1_reduced = functions.unreduced_to_reduced(PR_redux, pair1)
    p2_reduced = functions.unreduced_to_reduced(PR_redux, pair2)
    wt1, pos1, mt1 = functions.split_pair(p1_reduced)
    wt2, pos2, mt2 = functions.split_pair(p2_reduced)
# PR_pair = 'D30N-N88D'
# PR_pair = 'V32I-I47V'
# PR_pair = 'G48V-I54A'
# PR_pair = 'D30N-K45Q'
# PR_pair = 'I54A-V82A'

    pair1, pair2 = functions.split_pairs(PR_pair)

    p1_reduced = functions.unreduced_to_reduced(PR_redux, pair1)
    p2_reduced = functions.unreduced_to_reduced(PR_redux, pair2)

    wt1, pos1, mt1 = functions.split_pair(p1_reduced)
    wt2, pos2, mt2 = functions.split_pair(p2_reduced)

    ########
    gof_counts = 0

    gof_without_DMC_count = 0
    gof_with_DMC_count = 0
    gof_with_one_mut_count = 0

    ########
    rescue_count = 0

    rescue_without_DMC_count = 0
    rescue_with_DMC_count = 0
    rescue_with_one_mut_count = 0

    ########
    compensatory_counts = 0

    compensatory_without_DMC_count = 0
    compensatory_with_DMC_count = 0
    compensatory_with_one_mut_count = 0

    ########
    noncompensatory_counts = 0

    noncompensatory_without_DMC_count = 0
    noncompensatory_with_DMC_count = 0
    noncompensatory_with_one_mut_count = 0


    for PR_seq in PR_all_seq:
        result_merged = functions.calculate_dde_v2(p1_reduced, p2_reduced, PR_seq, PR_J, 1, 99)

        if result_merged is None:
            continue
        pair1_de, pair2_de, pair12_de, pair12_dde = result_merged

        if pair1_de < pair12_de and pair2_de < pair12_de and pair12_de > 0:
            gof_counts += 1


            if PR_seq[pos1 - 1] == wt1 and PR_seq[pos2 - 1] == wt2:
                gof_without_DMC_count += 1
            elif PR_seq[pos1 - 1] == mt1 and PR_seq[pos2 - 1] == mt2:
                gof_with_DMC_count += 1
            else:
                gof_with_one_mut_count += 1

        elif pair1_de < pair12_de and pair2_de < pair12_de:
            rescue_count += 1

            if PR_seq[pos1 - 1] == wt1 and PR_seq[pos2 - 1] == wt2:
                rescue_without_DMC_count += 1
            elif PR_seq[pos1 - 1] == mt1 and PR_seq[pos2 - 1] == mt2:
                rescue_with_DMC_count += 1
            else:
                rescue_with_one_mut_count += 1

        elif pair1_de < pair12_de or pair2_de < pair12_de:
            compensatory_counts += 1
        
            if PR_seq[pos1 - 1] == wt1 and PR_seq[pos2 - 1] == wt2:
                compensatory_without_DMC_count += 1
            elif PR_seq[pos1 - 1] == mt1 and PR_seq[pos2 - 1] == mt2:
                compensatory_with_DMC_count += 1
            else:
                compensatory_with_one_mut_count += 1
        else:
            noncompensatory_counts += 1

            if PR_seq[pos1 - 1] == wt1 and PR_seq[pos2 - 1] == wt2:
                noncompensatory_without_DMC_count += 1
            elif PR_seq[pos1 - 1] == mt1 and PR_seq[pos2 - 1] == mt2:
                noncompensatory_with_DMC_count += 1
            else:
                noncompensatory_with_one_mut_count += 1

    gof_with_DMC_ratio = gof_with_DMC_count / gof_counts
    rescue_with_DMC_ratio = rescue_with_DMC_count / rescue_count
    gof_rescue_with_DMC_ratio = (gof_with_DMC_count + rescue_with_DMC_count) / (gof_counts + rescue_count)
    noncompensatory_with_DMC_ratio = noncompensatory_with_DMC_count / noncompensatory_counts

    print('Gain of function with', PR_pair, gof_with_DMC_ratio)
    print('Rescue with', PR_pair, rescue_with_DMC_ratio)
    print('GOF+Rescue with', PR_pair, gof_rescue_with_DMC_ratio)
    print('Noncompensatory with', PR_pair, noncompensatory_with_DMC_ratio)

    min_pos = 1
    max_pos = 99
    pair1, pair2 = functions.split_pairs(PR_pair)

    p1_reduced = functions.unreduced_to_reduced(PR_redux, pair1)
    p2_reduced = functions.unreduced_to_reduced(PR_redux, pair2)

    wt1, pos1, mt1 = functions.split_pair(p1_reduced)
    wt2, pos2, mt2 = functions.split_pair(p2_reduced)

    gof_seqs = []
    gof_without_DMC = []
    gof_with_DMC = []

    rescue_seqs = []

    noncomp_seqs = []

    consensus_result = functions.calculate_delta_delta_e(p1_reduced, p2_reduced, PR_consensus_seq, PR_J, min_pos, max_pos)
    consensus_flip_relation = consensus_result[0] - consensus_result[1]

    for PR_seq in PR_all_seq:
        result_merged = functions.calculate_dde_v2(p1_reduced, p2_reduced, PR_seq, PR_J, min_pos, max_pos)

        if result_merged is None:
            continue
        pair1_de, pair2_de, pair12_de, pair12_dde = result_merged

        if pair1_de < pair12_de and pair2_de < pair12_de and pair12_de > 0:
            # gain of function
            gof_seqs.append(PR_seq)
            if PR_seq[pos1 - min_pos] == wt1 and PR_seq[pos2 - min_pos] == wt2:
                gof_without_DMC.append(PR_seq)

            if PR_seq[pos1 - min_pos] == mt1 and PR_seq[pos2 - min_pos] == mt2:
                gof_with_DMC.append(PR_seq)

        elif pair1_de < pair12_de and pair2_de < pair12_de:
            # rescue
            rescue_seqs.append(PR_seq)

        elif pair1_de < pair12_de or pair2_de < pair12_de:
            # compensatory
            continue
        else:
            # non-compensatory
            noncomp_seqs.append(PR_seq)

    #####################
    print(len(gof_seqs), "GoF sequences found.")
    p_SH_gofs = []
    for gof_seq in gof_seqs:
        p_SH = functions.calculate_double_mutant_probablity(gof_seq, p1_reduced, p2_reduced, PR_J, min_pos, max_pos)
        p_SH_gofs.append(p_SH)

    average_p_gof = sum(p_SH_gofs) / len(p_SH_gofs) if p_SH_gofs else 0
    print("Average p_SH for GoF sequences:", average_p_gof)

    ######################
    print(len(rescue_seqs), "rescue sequences found.")
    p_SH_rescues = []
    for rescue_seq in rescue_seqs:
        p_SH = functions.calculate_double_mutant_probablity(rescue_seq, p1_reduced, p2_reduced, PR_J, min_pos, max_pos)
        p_SH_rescues.append(p_SH)

    average_p_rescue = sum(p_SH_rescues) / len(p_SH_rescues) if p_SH_rescues else 0
    print("Average p_SH for rescue sequences:", average_p_rescue)
    #######################
    average_p_gof_rescue = (sum(p_SH_gofs) + sum(p_SH_rescues)) / (len(p_SH_gofs) + len(p_SH_rescues)) if (len(p_SH_gofs) + len(p_SH_rescues)) > 0 else 0
    print("Average p_SH for GoF + Rescue sequences:", average_p_gof_rescue)
    #######################
    print(len(noncomp_seqs), "non-compensatory sequences found.")
    p_SH_noncomps = []
    for noncomp_seq in noncomp_seqs:
        p_SH = functions.calculate_double_mutant_probablity(noncomp_seq, p1_reduced, p2_reduced, PR_J, min_pos, max_pos)
        p_SH_noncomps.append(p_SH)

    average_p_noncomp = sum(p_SH_noncomps) / len(p_SH_noncomps) if p_SH_noncomps else 0
    print("Average p_SH for non-compensatory sequences:", average_p_noncomp)
##

    # Calculate the actual_p values
    gof_actual_p = gof_with_DMC_count / gof_counts if gof_counts > 0 else 0
    rescue_actual_p = rescue_with_DMC_count / rescue_count if rescue_count > 0 else 0
    gof_rescue_actual_p = (gof_with_DMC_count + rescue_with_DMC_count) / (gof_counts + rescue_count) if (gof_counts + rescue_count) > 0 else 0
    noncomp_actual_p = noncompensatory_with_DMC_count / noncompensatory_counts if noncompensatory_counts > 0 else 0

    # Append data for each epistasis subset
    csv_data.append([PR_pair, 'Gain_of_function', gof_counts, average_p_gof, gof_actual_p])
    csv_data.append([PR_pair, 'rescue', rescue_count, average_p_rescue, rescue_actual_p])
    csv_data.append([PR_pair, 'rescue+gof', gof_counts + rescue_count, average_p_gof_rescue, gof_rescue_actual_p])
    csv_data.append([PR_pair, 'non_compensatory', noncompensatory_counts, average_p_noncomp, noncomp_actual_p])

# Write the data to a CSV file
with open('protease_subset_probabilities.csv', 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow(['mutation_pair', 'epistasis_subset', 'num_seqs', 'average_p', 'actual_p'])
    csv_writer.writerows(csv_data)

print("CSV file 'protease_subset_probabilities.csv' created successfully.")

Gain of function with D30N-N88D 0.5897435897435898
Rescue with D30N-N88D 0.14842903575297942
GOF+Rescue with D30N-N88D 0.19807692307692307
Noncompensatory with D30N-N88D 0.0005197505197505198
117 GoF sequences found.
Average p_SH for GoF sequences: 0.5945075070021693
923 rescue sequences found.
Average p_SH for rescue sequences: 0.12256968846902588
Average p_SH for GoF + Rescue sequences: 0.1756626930540045
1924 non-compensatory sequences found.
Average p_SH for non-compensatory sequences: 0.0007638448486228985
Gain of function with V32I-I47V 0.6571428571428571
Rescue with V32I-I47V 0.2087378640776699
GOF+Rescue with V32I-I47V 0.36012861736334406
Noncompensatory with V32I-I47V 0.002455905336012503
105 GoF sequences found.
Average p_SH for GoF sequences: 0.719022064194226
206 rescue sequences found.
Average p_SH for rescue sequences: 0.15444471549538485
Average p_SH for GoF + Rescue sequences: 0.34505764672811257
4479 non-compensatory sequences found.
Average p_SH for non-compensatory s

In [ ]:
import csv
min_pos = 39
max_pos = 226
RT_pairs=['K101E-G190S', 'K101E-G190A', 'K103N-P225H', 'L100I-K103N', 'K101P-K103S', 'Y181C-H221Y', 'K103S-G190A', 'K103S-P225H', 'L100I-K103R', 'V108I-H221Y']
RT_pairs.extend([
    'F116Y-Q151M',
    'M41L-T215Y',
    'V75I-I132L',
    'K70R-K219E',
    'D67N-K219Q',
    'K70R-K219Q',
    'L210W-T215Y',
    'F116Y-Q151L',
    'K65R-S68N',
    'V75I-F77L'
])
# Define the epistasis subsets
epistasis_subsets = ['Gain_of_function', 'rescue', 'rescue+gof', 'non_compensatory']

# Prepare the data for the CSV
csv_data = []

for RT_pair in RT_pairs:
    print("Processing RT pair:", RT_pair)
    pair1, pair2 = functions.split_pairs(RT_pair)
    p1_reduced = functions.unreduced_to_reduced(RT_redux, pair1)
    p2_reduced = functions.unreduced_to_reduced(RT_redux, pair2)
    wt1, pos1, mt1 = functions.split_pair(p1_reduced)
    wt2, pos2, mt2 = functions.split_pair(p2_reduced)

    ########
    gof_counts = 0
    gof_without_DMC_count = 0
    gof_with_DMC_count = 0
    gof_with_one_mut_count = 0

    ########
    rescue_count = 0
    rescue_without_DMC_count = 0
    rescue_with_DMC_count = 0
    rescue_with_one_mut_count = 0

    ########
    compensatory_counts = 0
    compensatory_without_DMC_count = 0
    compensatory_with_DMC_count = 0
    compensatory_with_one_mut_count = 0

    ########
    noncompensatory_counts = 0
    noncompensatory_without_DMC_count = 0
    noncompensatory_with_DMC_count = 0
    noncompensatory_with_one_mut_count = 0

    gof_seqs = []
    rescue_seqs = []
    noncomp_seqs = []

    for RT_seq in RT_all_seq:
        result_merged = functions.calculate_dde_v2(p1_reduced, p2_reduced, RT_seq, RT_J, min_pos, max_pos)

        if result_merged is None:
            continue
        pair1_de, pair2_de, pair12_de, pair12_dde = result_merged

        if pair1_de < pair12_de and pair2_de < pair12_de and pair12_de > 0:
            gof_counts += 1
            gof_seqs.append(RT_seq)
            if RT_seq[pos1 - min_pos] == wt1 and RT_seq[pos2 - min_pos] == wt2:
                gof_without_DMC_count += 1
            elif RT_seq[pos1 - min_pos] == mt1 and RT_seq[pos2 - min_pos] == mt2:
                gof_with_DMC_count += 1
            else:
                gof_with_one_mut_count += 1

        elif pair1_de < pair12_de and pair2_de < pair12_de:
            rescue_count += 1
            rescue_seqs.append(RT_seq)
            if RT_seq[pos1 - min_pos] == wt1 and RT_seq[pos2 - min_pos] == wt2:
                rescue_without_DMC_count += 1
            elif RT_seq[pos1 - min_pos] == mt1 and RT_seq[pos2 - min_pos] == mt2:
                rescue_with_DMC_count += 1
            else:
                rescue_with_one_mut_count += 1

        elif pair1_de < pair12_de or pair2_de < pair12_de:
            compensatory_counts += 1
            if RT_seq[pos1 - min_pos] == wt1 and RT_seq[pos2 - min_pos] == wt2:
                compensatory_without_DMC_count += 1
            elif RT_seq[pos1 - min_pos] == mt1 and RT_seq[pos2 - min_pos] == mt2:
                compensatory_with_DMC_count += 1
            else:
                compensatory_with_one_mut_count += 1
        else:
            noncompensatory_counts += 1
            noncomp_seqs.append(RT_seq)
            if RT_seq[pos1 - min_pos] == wt1 and RT_seq[pos2 - min_pos] == wt2:
                noncompensatory_without_DMC_count += 1
            elif RT_seq[pos1 - min_pos] == mt1 and RT_seq[pos2 - min_pos] == mt2:
                noncompensatory_with_DMC_count += 1
            else:
                noncompensatory_with_one_mut_count += 1

    # Calculate average probabilities
    p_SH_gofs = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, RT_J, min_pos, max_pos) for seq in gof_seqs]
    average_p_gof = sum(p_SH_gofs) / len(p_SH_gofs) if p_SH_gofs else 0

    p_SH_rescues = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, RT_J, min_pos, max_pos) for seq in rescue_seqs]
    average_p_rescue = sum(p_SH_rescues) / len(p_SH_rescues) if p_SH_rescues else 0

    average_p_gof_rescue = (sum(p_SH_gofs) + sum(p_SH_rescues)) / (len(p_SH_gofs) + len(p_SH_rescues)) if (len(p_SH_gofs) + len(p_SH_rescues)) > 0 else 0

    p_SH_noncomps = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, RT_J, min_pos, max_pos) for seq in noncomp_seqs]
    average_p_noncomp = sum(p_SH_noncomps) / len(p_SH_noncomps) if p_SH_noncomps else 0

    # Calculate the actual_p values
    gof_actual_p = gof_with_DMC_count / gof_counts if gof_counts > 0 else 0
    rescue_actual_p = rescue_with_DMC_count / rescue_count if rescue_count > 0 else 0
    gof_rescue_actual_p = (gof_with_DMC_count + rescue_with_DMC_count) / (gof_counts + rescue_count) if (gof_counts + rescue_count) > 0 else 0
    noncomp_actual_p = noncompensatory_with_DMC_count / noncompensatory_counts if noncompensatory_counts > 0 else 0

    # Append data for each epistasis subset
    csv_data.append([RT_pair, 'Gain_of_function', gof_counts, average_p_gof, gof_actual_p])
    csv_data.append([RT_pair, 'rescue', rescue_count, average_p_rescue, rescue_actual_p])
    csv_data.append([RT_pair, 'rescue+gof', gof_counts + rescue_count, average_p_gof_rescue, gof_rescue_actual_p])
    csv_data.append([RT_pair, 'non_compensatory', noncompensatory_counts, average_p_noncomp, noncomp_actual_p])
    print([RT_pair, 'Gain_of_function', gof_counts, average_p_gof, gof_actual_p])
    print([RT_pair, 'rescue', rescue_count, average_p_rescue, rescue_actual_p])
    print([RT_pair, 'rescue+gof', gof_counts + rescue_count, average_p_gof_rescue, gof_rescue_actual_p])
    print([RT_pair, 'non_compensatory', noncompensatory_counts, average_p_noncomp, noncomp_actual_p])
# Write the data to a CSV file
with open('reverse_transcriptase_subset_probabilities.csv', 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow(['mutation_pair', 'epistasis_subset', 'num_seqs', 'average_p', 'actual_p'])
    # print("Writing data to CSV file...",)
    csv_writer.writerows(csv_data)

print("CSV file 'reverse_transcriptase_subset_probabilities.csv' created successfully.")

Processing RT pair: K101E-G190S
['K101E-G190S', 'Gain_of_function', 100, 0.16256893311710996, 0.18]
['K101E-G190S', 'rescue', 1697, 0.06483708513159432, 0.08898055391868002]
['K101E-G190S', 'rescue+gof', 1797, 0.07027569659433866, 0.09404563160823595]
['K101E-G190S', 'non_compensatory', 13807, 0.0040592845882299645, 0.0028970811907003693]
Processing RT pair: K101E-G190A
['K101E-G190A', 'Gain_of_function', 534, 0.36496597808552517, 0.37640449438202245]
['K101E-G190A', 'rescue', 1169, 0.16171313678991278, 0.1924721984602224]
['K101E-G190A', 'rescue+gof', 1703, 0.22544597134766792, 0.2501467997651204]
['K101E-G190A', 'non_compensatory', 10402, 0.008471447544242922, 0.009901941934243414]
Processing RT pair: K103N-P225H
['K103N-P225H', 'Gain_of_function', 148, 0.4368802615735587, 0.5135135135135135]
['K103N-P225H', 'rescue', 28, 0.29440107944559185, 0.42857142857142855]
['K103N-P225H', 'rescue+gof', 176, 0.41421311896229124, 0.5]
['K103N-P225H', 'non_compensatory', 639, 0.001438512550581506